<a href="https://colab.research.google.com/github/Hassanmufezshaikh/AI-Agents/blob/main/ParallelAgentsArchitecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-adk google-genai

  Using cached google_genai-2.5.0-py3-none-any.whl.metadata (52 kB)


In [ ]:
import google.adk
print(google.adk.__version__)

2.0.0


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print(" Tunnel Components imported successfully")



 Tunnel Components imported successfully


In [ ]:
import os
from google.colab import userdata
userdata.get('gemeni')

try:
  GOOGLE_API_KEY = userdata.get('gemeni')
  os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
  print("Gemini API Key Setup Complete")
except Exception as e :
  print("Authencation Error: Please add 'GEMENI_API_KEY' to your kaggale secrets, Details : {e}")

Gemini API Key Setup Complete


In [ ]:
from google.adk.agents import (
    Agent,
    SequentialAgent,
    ParallelAgent,
    LoopAgent
)

from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types

print("ADK components imported successfully.")

ADK components imported successfully.


In [ ]:
import google.adk.tools as tools

print(dir(tools))

['APIHubToolset', 'AgentTool', 'AgentTool', 'Any', 'ApiRegistry', 'AuthToolArguments', 'BaseTool', 'DiscoveryEngineSearchTool', 'ExampleTool', 'FunctionTool', 'FunctionTool', 'LongRunningFunctionTool', 'MCPToolset', 'McpToolset', 'SearchResultMode', 'TYPE_CHECKING', 'ToolContext', 'TransferToAgentTool', 'VertexAiSearchTool', '_LAZY_MAPPING', '__all__', '__builtins__', '__cached__', '__dir__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_automatic_function_calling_util', '_forwarding_artifact_service', '_function_parameter_parse_util', '_function_tool_declarations', 'agent_tool', 'base_tool', 'base_toolset', 'computer_use', 'enterprise_web_search', 'exit_loop', 'function_tool', 'get_user_choice', 'google_maps_grounding', 'google_search', 'google_search', 'google_search_tool', 'importlib', 'load_artifacts', 'load_memory', 'logging', 'preload_memory', 'set_model_response_tool', 'sys', 'tool_configs', 'tool_confirmation', 'tool_co

In [ ]:
from google.genai import types

retry_config=types.HttpRetryOptions(
attempts=5, # Maximum retry attempts
exp_base=7, # Delay multiplier
initial_delay=1, # Initial delay before first retry (in seconds)
http_status_codes=[429, 500, 503, 504]
)

In [ ]:
# Research Agent: Its job is to use the google_search tool and present findings.

tech_researcher = Agent(
    name="TechResearcher",

    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),

    instruction="""
    You are a specialized research agent for technology. To answere the user query, you must use the 'google_search' tool to find the main companies involved, and the potential impact keep the report very concise (100 words).
    """,
    tools=[google_search],
    output_key="tech_research"
)

print("tech_researcher created.")

tech_researcher created.


In [ ]:
# Research Agent: Its job is to use the google_search tool and present findings.

health_researcher = Agent(
    name="HealthResearcher",

    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),

    instruction="""
    You are a specialized research agent for health. To answer the user query, you must use the 'google_search' tool to find the main companies involved, and the potential impact. Keep the report very concise (100 words).
    """,
    tools=[google_search],
    output_key="health_research"
)

print("health_researcher created.")

health_researcher created.


In [ ]:
# Research Agent: Its job is to use the google_search tool and present findings.

finance_researcher = Agent(
    name="FinanceResearcher",

    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),

    instruction="""
    You are a specialized research agent for finance. To answer the user query, you must use the 'google_search' tool to find the main companies involved, and the potential impact. Keep the report very concise (100 words).
    """,
    tools=[google_search],
    output_key="finance_research"
)

print("finance_researcher created.")

finance_researcher created.


In [ ]:
# Aggregator Agent: Its job is to synthesize results after a parallel step.

aggregator_agent = Agent(
    name="Aggregator",

    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),

    instruction="""
    You are an aggregator agent. Your task is to synthesize and combine information from various sources provided by other agents. Create a cohesive summary of the findings.
    """,
    # The aggregator agent might not need external tools if its role is purely to process internal outputs.
    # However, if it needs to perform further search based on aggregated results, google_search could be added.
    tools=[],
    output_key="aggregator_results"
)

print("Aggregator agent created.")

Aggregator agent created.


In [ ]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher]
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent]
)

print("Parallel and Sequential Agents created.")

Parallel and Sequential Agents created.


/tmp/ipykernel_2669/400011055.py:2: DeprecationWarning: ParallelAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  parallel_research_team = ParallelAgent(
/tmp/ipykernel_2669/400011055.py:8: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  root_agent = SequentialAgent(


In [ ]:
runner = InMemoryRunner(agent=root_agent)
print("InMemoryRunner created.")
response =  await runner.run_debug(
"run the daily executive breifing on tech, health and finance")

InMemoryRunner created.
FinanceResearcher > **Tech:** The U.S. is launching a financing initiative to boost AI exports, aiming to expand global demand for American AI technologies through loans and guarantees. This strategy complements existing export controls on advanced semiconductors to countries like China, seeking to solidify U.S. leadership in AI by fostering global adoption. Nvidia's performance is being closely watched, with discussions around "agentic AI" and its impact on compute capacity and profitability. GigaCloud Technology will participate in the Jefferies Software, Internet & AI Conference, and Arlo Technologies is set to present at the William Blair Growth Stock Conference.

**Health:** A hantavirus outbreak serves as a reminder for improved pandemic preparedness, emphasizing transparent communication and preventive measures like indoor air quality improvements and masks. The U.S. Preventive Services Task Force (USPSTF) is facing changes, with reports of the Trump admi

ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)